In [26]:
import pandas as pd
import re
import os
import sys

def load_knowledge_base(excel_path):
    """
    بارگذاری اکسل:
    - lexicon: دیکشنری ساده {کلمه: آوا} برای پردازش متن
    - alphabet: حروف الفبا برای هجی کردن
    - master_db: دیکشنری جامع {کلمه: {تمام ویژگی‌ها}} برای DictLookup
    """
    lexicon = {}   
    alphabet = {}
    master_db = {} # پایگاه داده جامع برای جستجوی ویژگی‌ها
    
    try:
        if not os.path.exists(excel_path):
            raise FileNotFoundError(f"فایل '{excel_path}' یافت نشد.")

        df = pd.read_excel(excel_path)
        # حذف فاصله‌های اضافی از نام ستون‌ها
        df.columns = [c.strip() for c in df.columns]

        for _, row in df.iterrows():
            word = str(row['کلمه']).strip()
            raw_phonetic = str(row['نمایش آوایی']).strip()
            pos = str(row['POS']).strip()
            
            # ذخیره تمام اطلاعات سطر در master_db
            # تبدیل کل سطر به دیکشنری برای دسترسی راحت به تمام ویژگی‌ها
            master_db[word] = row.to_dict()

            # --- منطق حروف الفبا (برای استفاده در هجی کردن) ---
            if pos == 'Al':
                if 'alef' in raw_phonetic or 'eyn' in raw_phonetic:
                    if not raw_phonetic.startswith('?'):
                        final_alpha = "?" + raw_phonetic
                    else:
                        final_alpha = raw_phonetic
                else:
                    final_alpha = raw_phonetic.replace('?', '')
                
                alphabet[word] = final_alpha
            
            # --- منطق کلمات (Lexicon) ---
            else:
                lexicon[word] = raw_phonetic

        print(f"--> بارگذاری: {len(lexicon)} کلمه و {len(alphabet)} حرف الفبا.")
        return lexicon, alphabet, master_db

    except Exception as e:
        print(f"خطا در بارگذاری: {e}")
        sys.exit(1)

def DictLookup(word, att, val_container, database):
    """
    شبیه‌سازی تابع: BOOLEAN DictLookup (string word, ATTTYPE att, (VOID *) val)
    
    Parameters:
        word (str): کلمه مورد نظر
        att (str): نام ستون یا ویژگی (مثلاً 'POS' یا 'نمایش آوایی')
        val_container (list): یک لیست خالی که به عنوان اشاره‌گر عمل می‌کند.
                              مقدار پیدا شده درون این لیست ریخته می‌شود.
        database (dict): دیکشنری جامع کلمات (master_db)
        
    Returns:
        bool: True اگر کلمه و ویژگی پیدا شد، False در غیر این صورت.
    """
    if word in database:
        word_data = database[word]
        if att in word_data:
            # مقدار را درون کانتینر (که نقش اشاره‌گر را دارد) قرار می‌دهیم
            # ابتدا لیست را خالی می‌کنیم تا مطمئن شویم داده قبلی پاک شده
            val_container.clear()
            val_container.append(word_data[att])
            return True
    
    return False

def clean_text(text):
    return re.sub(r'[!@#$%\^&*\\(\\)_+\={}\\[\\]:;"\'<>,.?/|\\«»]', ' ', text)

def spell_out_word(word, alphabet_map):
    spelled = []
    for char in word:
        if char in alphabet_map:
            spelled.append(alphabet_map[char])
        elif char != '\u200c': 
            spelled.append(char)
            
    if not spelled:
        return ""
    return "\\" + " ".join(spelled) + "\\"

def process_text_file(input_file, output_file, lexicon_db, alphabet_map):
    NOUNS_NEEDING_EZAFE = ["الگوریتم", "رایانش", "مدل", "روش", "پردازش"]
    STOP_WORDS = ["و", "یا", "در", "به", "از", "که", ".", "،"]

    try:
        # ایجاد فایل ورودی نمونه اگر وجود نداشت
        if not os.path.exists(input_file):
            with open(input_file, 'w', encoding='utf-8') as f:
                f.write("رایانش کوانتومی") 
        
        with open(input_file, 'r', encoding='utf-8') as f:
            content = f.read()

        raw_words = content.split()
        output_phonetics = []

        for i, word in enumerate(raw_words):
            clean_w = re.sub(r'[!@#$%\^&*\(\)_+\={}\[\]:;"\'<>,.?/|\\«»]', '', word)
            
            if not clean_w:
                continue

            if clean_w in lexicon_db:
                phonetic = lexicon_db[clean_w]
            else:
                phonetic = spell_out_word(clean_w, alphabet_map)

            need_ezafe = False
            if word.endswith('ِ') or word.endswith('\u0650'):
                need_ezafe = True
            elif clean_w in NOUNS_NEEDING_EZAFE:
                if i + 1 < len(raw_words):
                    next_word_raw = raw_words[i+1]
                    clean_next = re.sub(r'[!@#$%\^&*\(\)_+\={}\[\]:;"\'<>,.?/|\\«»]', '', next_word_raw)
                    if clean_next not in STOP_WORDS:
                         need_ezafe = True

            if need_ezafe:
                if not phonetic.startswith('\\'):
                    phonetic += "-e"

            output_phonetics.append(phonetic)

        final_output = " ".join(output_phonetics)

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(final_output)

        print(f"خروجی فایل متنی ایجاد شد: {final_output[:100]}...")

    except Exception as e:
        print(f"خطا در پردازش متن: {e}")


if __name__ == "__main__":
    excel_source = "Flexicon.xlsx"
    input_txt = "input.txt"
    output_txt = "output_phonetic.txt"

    # 1. بارگذاری داده‌ها
    lexicon_db, alphabet_map, master_db = load_knowledge_base(excel_source)
    
    if lexicon_db:
        # 2. پردازش فایل متنی 
        process_text_file(input_txt, output_txt, lexicon_db, alphabet_map)

        # 3. تست تابع DictLookup
        #----------------------
        print("\n--- تست تابع DictLookup ---")
        
        test_word = "رایانش" 
        target_att = "POS"  

        val_result = [] 
        
        found = DictLookup(test_word, target_att, val_result, master_db)
        
        if found:
            # مقدار استخراج شده در اولین خانه لیست قرار دارد
            extracted_value = val_result[0]
            print('موفقیت:')
            print(f"ویژگی '{target_att}' برای کلمه '{test_word}' برابر است با:")
            print(extracted_value)
            print('----------------------')
        else:
            print('شکست:')
            print(f" کلمه '{test_word}' یا ویژگی '{target_att}' یافت نشد.")
            print('----------------------')

        val_result2 = []
        if DictLookup("کیوبیت", "نمایش آوایی", val_result2, master_db):
             print(f"آوانگاری 'رایانش': {val_result2[0]}")
        #----------------------
        #  پایان تست تابع DictLookup 


--> بارگذاری: 50 کلمه و 38 حرف الفبا.
خروجی فایل متنی ایجاد شد: rAyAneS-e \kAf vAv ?alef nun te vAv mim ye\ va gugel va Algoritm-e Sor...

--- تست تابع DictLookup ---
موفقیت:
ویژگی 'POS' برای کلمه 'رایانش' برابر است با:
N1
----------------------
آوانگاری 'رایانش': kiyubit
